In [2]:
import duckdb
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
import os

In [3]:
WAREHOUSE_TEST = "./test_duckdb_data/gtftest22.duckdb"

con = duckdb.connect(WAREHOUSE_TEST)
con.execute("SET TimeZone='UTC';")

con.sql("SHOW TABLES").df()

,name
0,agency
1,calendar
2,calendar_dates
3,delays_with_support_columns
4,feed_info
5,per_event_corrected
6,routes
7,shapes
8,stop_times
9,stop_with_delay


In [4]:
con.execute("""
CREATE OR REPLACE VIEW per_event_corrected AS
WITH first_stop AS (
    SELECT trip_id, MIN(stop_sequence) AS first_seq
    FROM trips_updates
    GROUP BY trip_id
),

trip_start AS (
    SELECT
        tu.trip_id,
        tu.arrival_dt AS trip_start_utc,
        st.arrival_time_sec AS trip_start_sec
    FROM trips_updates tu
    JOIN first_stop fs 
        ON tu.trip_id = fs.trip_id AND tu.stop_sequence = fs.first_seq
    JOIN stop_times st 
        ON tu.trip_id = st.trip_id AND tu.stop_sequence = st.stop_sequence
    WHERE tu.arrival_dt IS NOT NULL AND st.arrival_time_sec IS NOT NULL
),

full_data AS (
    SELECT
        tu.trip_id,
        tu.stop_sequence,
        st.stop_id,
        CAST(tu.arrival_dt AS TIMESTAMP WITH TIME ZONE) AS arrival_dt_utc,
        st.arrival_time_sec,
        ts.trip_start_utc,
        ts.trip_start_sec
    FROM trips_updates tu
    JOIN stop_times st 
        ON tu.trip_id = st.trip_id AND tu.stop_sequence = st.stop_sequence
    JOIN trip_start ts 
        ON tu.trip_id = ts.trip_id
)

SELECT
    trip_id,
    stop_sequence,
    stop_id,
    arrival_dt_utc,
    trip_start_utc + (arrival_time_sec - trip_start_sec) * INTERVAL '1 second' AS scheduled_ts_utc,
    EXTRACT(EPOCH FROM (
        arrival_dt_utc - (
            trip_start_utc + (arrival_time_sec - trip_start_sec) * INTERVAL '1 second'
        )
    )) / 60.0 AS delay_min
FROM full_data
WHERE arrival_time_sec IS NOT NULL;
""")


In [5]:
con.sql("""
SELECT * 
FROM per_event_corrected
ORDER BY arrival_dt_utc DESC
LIMIT 100;
""").df()


,trip_id,stop_sequence,stop_id,arrival_dt_utc,scheduled_ts_utc,delay_min
0,6452593-T1_R_57_VT110_23:00-SETP2025-L1-Semain...,4,2535,2025-09-11 21:07:00+00:00,2025-09-11 21:07:00,0.000000
1,6452593-T1_R_57_VT110_23:00-SETP2025-L1-Semain...,3,2543,2025-09-11 21:06:00+00:00,2025-09-11 21:06:00,0.000000
2,6452593-T1_R_57_VT110_23:00-SETP2025-L1-Semain...,2,2531,2025-09-11 21:04:00+00:00,2025-09-11 21:04:00,0.000000
3,6170755-L3_A_48__22:54-JANV2025-L3-Semaine-09,5,2800,2025-09-11 21:03:00+00:00,2025-09-11 21:03:00,0.000000
4,6452593-T1_R_57_VT110_23:00-SETP2025-L1-Semain...,1,2529,2025-09-11 21:02:00+00:00,2025-09-11 21:02:00,0.000000
...,...,...,...,...,...,...
95,6428493-61_A_50_6104_21:30-PROJET2025-61-Semai...,34,266,2025-09-11 20:36:06+00:00,2025-09-11 20:36:29,-0.383333
96,5839180-Exp4_R_97_EXP401_22:10-PROJET2025-Exp4...,19,2835,2025-09-11 20:36:05+00:00,2025-09-11 20:35:39,0.433333
97,6428363-60_A_50_6001_21:10-PROJET2025-60-Semai...,18,593,2025-09-11 20:36:00+00:00,2025-09-11 20:36:00,0.000000
98,6452590-T1_R_57_VT122_22:15-SETP2025-L1-Semain...,12,2523,2025-09-11 20:36:00+00:00,2025-09-11 20:36:01,-0.016667


In [6]:
con.execute("""
CREATE OR REPLACE VIEW stop_with_delay AS
SELECT
    s.stop_id,
    s.stop_name,
    s.stop_lat,
    s.stop_lon,
    ROUND(AVG(pe.delay_min), 2) AS avg_delay_min
FROM stops s
LEFT JOIN per_event_corrected pe
    ON s.stop_id = pe.stop_id
GROUP BY s.stop_id, s.stop_name, s.stop_lat, s.stop_lon;
""")


In [7]:
con.sql("""
SELECT * 
FROM stop_with_delay
ORDER BY avg_delay_min DESC
LIMIT 700;
""").df()


,stop_id,stop_name,stop_lat,stop_lon,avg_delay_min
0,2771,Parc Phoenix,43.669416,7.219098,1.27
1,2622,Ferber,43.676896,7.228362,1.27
2,2767,Grand Arénas,43.669944,7.212342,1.27
3,2624,Cassin / Kirchner,43.672277,7.224127,1.27
4,1173,Route d'Aspremont,43.749012,7.247345,1.13
...,...,...,...,...,...
695,1266,Sainte-Hélène,43.685704,7.235695,NaN
696,1005,Papyrus,43.705006,7.294971,NaN
697,948,Avenue Mont Alban,43.701295,7.295847,NaN
698,8135,Parc Rivièra,43.733764,7.348939,NaN


In [8]:
con.sql("""
    SELECT 
        MIN(delay_min) AS min_delay,
        MAX(delay_min) AS max_delay,
        AVG(delay_min) AS avg_delay
    FROM per_event_corrected;
""").df()


,min_delay,max_delay,avg_delay
0,-4.25,1.266667,-0.309561


In [9]:


df = con.execute("""
    SELECT
        route_id,
        ROUND(AVG(delay_min), 2) AS avg_delay_min,
        COUNT(*) AS num_events
    FROM vehicle_positions vp
    LEFT JOIN delays_with_support_columns d
        ON vp.trip_id = d.trip_id
    WHERE vp.timestamp IS NOT NULL
    GROUP BY route_id
    ORDER BY avg_delay_min DESC
    LIMIT 20
""").df()

df




,route_id,avg_delay_min,num_events
0,15,0.44,10
1,63,0.15,25
2,51,0.00,4
3,21,-0.04,10
4,32,-0.08,4
5,05,-0.27,15
6,62,-0.33,18
7,chouette:Line:4a8dc505-72e7-4b72-b186-a5b46e94...,-0.52,24
8,17,-0.53,26
9,09,-0.57,22


In [ ]:
#question 3 carte_arret

df = con.sql("""
    SELECT
        s.stop_id,
        s.stop_name,
        s.stop_lat,
        s.stop_lon,
        ROUND(AVG(pe.delay_min), 2) AS avg_delay_min
    FROM stops s
    LEFT JOIN per_event_corrected pe
        ON s.stop_id = pe.stop_id
    GROUP BY s.stop_id, s.stop_name, s.stop_lat, s.stop_lon
    ORDER BY avg_delay_min DESC
""").df()

df

,stop_id,stop_name,stop_lat,stop_lon,avg_delay_min
0,2624,Cassin / Kirchner,43.672277,7.224127,1.27
1,2767,Grand Arénas,43.669944,7.212342,1.27
2,2622,Ferber,43.676896,7.228362,1.27
3,2771,Parc Phoenix,43.669416,7.219098,1.27
4,1173,Route d'Aspremont,43.749012,7.247345,1.13
...,...,...,...,...,...
4480,6213,Plesent,43.766809,7.221927,NaN
4481,21620,Parc Alpha Loup,44.111776,7.295248,NaN
4482,21314,Village,44.073328,7.251618,NaN
4483,place_GENDAR,Gendarmerie,44.078159,7.249851,NaN
